In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import shape
from shapely import wkt
from shapely.ops import unary_union
from hda import Client as hda_client
from pystac_client import Client

In [6]:
from datetime import datetime, timedelta

In [ ]:
from hda import Configuration
conf = Configuration(user='USERNAME_HERE', password='PASSWORD_HERE')

In [8]:
DATA_DIR = '../data/'

In [5]:
tile_dates = {
    # '30UXB': '2025-08-10T11:07:01.024000Z',
    '31UCS': '2025-08-25T10:56:41.025000Z',
    # '30UVD': '2025-08-16T11:21:19.024000Z',
    '31UCT': '2025-08-15T10:56:41.025000Z'
}

In [6]:
def download_image_hda(tileId, startdate, enddate, download_dir):
    c = hda_client(conf)
    
    q = {
        "dataset_id": "EO:ESA:DAT:SENTINEL-2",
        "startdate": f"{startdate}",
        "enddate": f"{enddate}",
        "processingLevel": "S2MSI1C",
        "tileId": f"{tileId}"
    }
    os.makedirs(download_dir, exist_ok=True)
    
    try:
        r = c.search(q)
        r[0].download(download_dir)
        print(f"Downloaded for tile {tileId}")
    except Exception as e:
        print(f"Error downloading for tile {tileId}")
        print(e)

In [7]:
for tileId, datetime_str in tile_dates.items():
    download_dir = os.path.join(DATA_DIR, f's2_downloads/{tileId}')
    date_str = datetime_str[:10].replace('-', '')

    if os.path.isdir(download_dir):
        existing_files = os.listdir(download_dir)
        if any(date_str in fname for fname in existing_files):
            print(f"Skipping {tileId}: file already exists.")
            continue  # skip if already downloaded
    
    startdate = datetime_str[:10]
    startdate_elem = datetime.strptime(startdate, '%Y-%m-%d')
    enddate_elem = startdate_elem + timedelta(days=1)
    enddate = enddate_elem.strftime('%Y-%m-%d')
    print(f"Downloading {tileId} for {date_str}")
    download_image_hda(tileId, startdate, enddate, download_dir)

Skipping 30UXB: file already exists.
Skipping 31UCS: file already exists.


Downloaded for tile 30UVD
Skipping 31UCT: file already exists.


In [8]:
import zipfile

In [9]:
for tileId in tile_dates:
    print(f"Unzipping for tile {tileId}")
    download_dir = os.path.join(DATA_DIR, f's2_downloads/{tileId}/')
    safe_fns = [x for x in os.listdir(download_dir) if x.endswith('.SAFE')]

    if len(safe_fns) > 0:  # already unzipped
        continue
    
    try:
        zip_fn = [x for x in os.listdir(download_dir) if x.endswith('.zip')][0]
        zip_fp = os.path.join(download_dir, zip_fn)
        with zipfile.ZipFile(zip_fp) as zf:
            zf.extractall(download_dir)
    except Exception as e:
        print(f"Error with tile {tileId}: {e}")

Unzipping for tile 30UXB
Unzipping for tile 31UCS
Unzipping for tile 30UVD
Error with tile 30UVD: File is not a zip file
Unzipping for tile 31UCT
